# Two-tower model — standalone inference notebook

Self-contained batch inference: **no `import two_tower`**. All library code is inlined below.

## Prerequisites
Training artifacts must exist under `paths.artifacts_base`:
- `artifacts/user_tower/user_tower_state.pt`
- `artifacts/vocab_artifact/vocab_artifact.pkl`
- `artifacts/client_embeddings/client_embeddings.parquet`

In [ ]:
import subprocess
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'requirements-mlops.txt').exists():
            return p
        if (p / 'configs' / 'train.yaml').exists():
            return p
    return cwd


_REPO = _find_repo_root()
_REQ = _REPO / 'requirements-mlops.txt'
if _REQ.is_file():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(_REQ)])
else:
    subprocess.check_call(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'torch>=2.2.0', 'pandas>=2.0.0', 'numpy>=1.26.0',
            'scikit-learn>=1.4.0',
            'pyarrow>=15.0.0', 's3fs>=2024.1.0',
            'polars>=0.20.0', 'PyYAML>=6.0.0',
        ]
    )
print('Dependencies OK.')


## Config
Edit the YAML below — same schema as `configs/infer.yaml`.


In [ ]:
import yaml

INFER_CONFIG_YAML = r'''paths:
  infer: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/test_data_final/20260501_full/"
  artifacts_base: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/3_Client/"

features:
  device_id_col: "device_id"
  user_feature_cols: []
  user_multi_cols: []

infer:
  topk_clients: 3
  infer_stream_batch_rows: 65536
  num_physical_gpus: 4
  workers_per_gpu: 3
  ranking_output: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/user_client_rankings_0501/"
  rank_user_batch: 2048
  client_chunk: 2048
  use_amp: true
  amp_dtype: "float16"
  user_tower_backend: "pytorch"
  user_tower_onnx_uri: null
  trt_fp16_enable: true
  trt_engine_cache_enable: true
  trt_engine_cache_path: null
  output_min_rows_per_part: 100000
  output_parquet_compression: "zstd"
  debug_cuda: false
  # Optional run-size controls for quick smoke tests (null = no limit).
  max_files: null
  max_users_per_file: null

extra: {}
'''
print('Loaded INFER_CONFIG_YAML (edit this cell if needed).')


## Backend note
- If `infer.user_tower_backend: pytorch`, you **do not** need ONNX Runtime.
- If `infer.user_tower_backend: tensorrt_onnxruntime`, you must have **onnxruntime-gpu** installed.


In [ ]:
import yaml
import sys

_cfg_preview = yaml.safe_load(INFER_CONFIG_YAML) or {}
_backend = str((_cfg_preview.get('infer') or {}).get('user_tower_backend', 'pytorch')).lower()
if _backend == 'tensorrt_onnxruntime':
    try:
        import onnxruntime  # noqa: F401
        print('onnxruntime present ✔')
    except Exception:
        import subprocess
        py = (sys.version_info.major, sys.version_info.minor)
        print(f'onnxruntime missing (python={py[0]}.{py[1]}).')
        # onnxruntime-gpu wheels may not be published for very new Python versions (e.g. 3.12).
        # Try GPU package first, then CPU fallback, and if both fail instruct to switch backend.
        pkgs = ['onnxruntime-gpu', 'onnxruntime']
        ok = False
        for pkg in pkgs:
            try:
                print(f'Attempting install: {pkg} ...')
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
                import onnxruntime  # noqa: F401
                print(f'Installed {pkg} ✔')
                ok = True
                break
            except Exception as e:
                print(f'Install failed for {pkg}: {e!r}')
        if not ok:
            raise RuntimeError(
                'ONNX Runtime is not available in this environment. '
                'Either use a Python version with onnxruntime(-gpu) wheels (commonly 3.10/3.11), '
                'or switch INFER_CONFIG_YAML: infer.user_tower_backend to "pytorch".'
            )
else:
    print(f'backend={_backend} (no onnxruntime required)')


## Inlined library code (generated)


In [ ]:

# ======================================================================================
# Config dataclasses
# Source: configs.py
# ======================================================================================

from dataclasses import dataclass, field
from typing import Any


@dataclass(frozen=True)
class DataPaths:
    train: str
    val: str
    infer: str
    artifacts_base: str  # e.g. s3://.../two_tower_ml or local path


@dataclass(frozen=True)
class FeatureConfig:
    label_col: str
    device_id_col: str
    client_id_col: str
    user_feature_cols: list[str]
    client_feature_cols: list[str]
    user_multi_cols: list[str]
    client_multi_cols: list[str]


@dataclass(frozen=True)
class TrainConfig:
    experiment_name: str
    run_name: str | None
    seed: int

    batch_size: int
    epochs: int
    lr: float
    weight_decay: float

    embed_dim: int  # final embedding dim (e.g. 1024)
    user_mlp_hidden: list[int]

    min_count: int
    num_oov_buckets: int
    multi_max_tokens: int

    num_workers: int
    device: str  # "cuda" or "cpu"

    # If set, run a fixed number of optimizer steps per epoch (option B training).
    # When unset, an epoch iterates over the DataLoader once.
    steps_per_epoch: int | None = None

    # Optional: compute train metrics each epoch on a bounded sample of seen examples.
    # Set to 0/None to disable.
    train_eval_max_examples: int | None = None
    # Optional: per-client train metrics sample size (defaults to train_eval_max_examples if unset).
    train_client_eval_max_examples: int | None = None

    # Performance / distributed (all optional; defaults preserve current behavior)
    # If torch.distributed env vars are set (WORLD_SIZE>1), training will use DDP automatically.
    dataloader_pin_memory: bool = True
    dataloader_persistent_workers: bool = True
    dataloader_prefetch_factor: int = 2

    # Optional compile (PyTorch 2.x). Keeps FP32 math; can be disabled if unstable.
    torch_compile: bool = False
    torch_compile_mode: str = "reduce-overhead"

    # Optional per-client downsampling (structured version of notebook Step 3a).
    # Applied inside ``load_train_validation_frames`` so it works the same for notebook and DDP.
    downsample_train: bool = False
    downsample_val: bool = False
    # Target negatives per positive within each client (None disables ratio balancing).
    downsample_neg_per_pos: int | None = 10
    # After ratio balancing, cap every client to the same row count (min across clients).
    downsample_equalize_client_rows: bool = True
    downsample_random_state: int = 42

    # Batch-level balancing (does NOT change dataset size; affects only sampling into batches).
    # When enabled, batches are constructed to match global per-client frequency and enforce a
    # per-client negative-to-positive ratio (e.g. 3 means 3 negatives per positive).
    batch_balance: bool = False
    batch_balance_neg_per_pos: int = 3

    # Hash embedding init for categorical columns (reference: PRETRAINED_EMB_DIM)
    pretrained_emb_dim: int = 128
    pretrained_cat_emb_dim: int = 64
    freeze_pretrained_base: bool = True

    # Multi-value categorical pooling in towers (reference: MULTI_CAT_POOL)
    multi_cat_pool: str = "mean"

    # Client MLP hidden layers
    client_mlp_hidden: list[int] = field(default_factory=lambda: [256, 256])


@dataclass(frozen=True)
class InferenceConfig:
    topk_clients: int
    infer_stream_batch_rows: int

    # worker pool / GPU sharding
    num_physical_gpus: int
    workers_per_gpu: int

    # outputs
    ranking_output: str

    # batching / perf (reference: RANK_USER_BATCH, CLIENT_CHUNK, etc.)
    rank_user_batch: int = 4096
    client_chunk: int = 8192
    use_amp: bool = True
    amp_dtype: str = "float16"
    output_min_rows_per_part: int = 100_000
    output_parquet_compression: str = "zstd"
    debug_cuda: bool = False
    # User tower runtime backend:
    # - "pytorch": existing nn.Module execution (default)
    # - "tensorrt_onnxruntime": ONNX Runtime with TensorRT EP (falls back to CUDA/CPU EP)
    user_tower_backend: str = "pytorch"
    # Required when user_tower_backend="tensorrt_onnxruntime".
    # Accepts local path or s3:// URI to an ONNX model that takes:
    #   user_cat(int64), user_num(float32), user_multi(int64) -> user embedding(float32)
    user_tower_onnx_uri: str | None = None
    trt_fp16_enable: bool = True
    trt_engine_cache_enable: bool = True
    trt_engine_cache_path: str | None = None

    # Optional run-size controls for quick smoke tests.
    # - max_files: only process first N parquet inputs under paths.infer
    # - max_users_per_file: stop after ranking N unique users per parquet file
    max_files: int | None = None
    max_users_per_file: int | None = None


@dataclass(frozen=True)
class InferPaths:
    """Minimal paths for a standalone inference job (`configs/infer.yaml`)."""

    infer: str
    artifacts_base: str


@dataclass(frozen=True)
class InferJobConfig:
    paths: InferPaths
    infer: InferenceConfig


@dataclass(frozen=True)
class DataLoadConfig:
    """Optional preprocessing when loading train/val (matches reference notebook)."""

    # Per-row join: ``client_metadata`` Parquet (same schema as pipeline ``client_metadata`` table)
    merge_client_metadata: bool = False
    client_metadata_uri: str | None = None

    inject_single_client_metadata: bool = False
    single_client_metadata_uri: str | None = None
    single_client_row_filter: dict[str, Any] | None = None
    single_client_features_hardcoded: dict[str, Any] | None = None


@dataclass(frozen=True)
class PipelineConfig:
    paths: DataPaths
    features: FeatureConfig
    train: TrainConfig
    infer: InferenceConfig
    data_load: DataLoadConfig = field(default_factory=DataLoadConfig)

    mlflow_tracking_uri: str | None = None
    extra: dict[str, Any] | None = None

# ======================================================================================
# Config loader
# Source: config_loader.py
# ======================================================================================

from pathlib import Path
from typing import Any

import yaml



def _parse_inference_section(i: dict, *, path_label: str) -> InferenceConfig:
    for k in (
        "topk_clients",
        "infer_stream_batch_rows",
        "num_physical_gpus",
        "workers_per_gpu",
        "ranking_output",
    ):
        if k not in i or i[k] is None:
            raise KeyError(f"infer.{k} is required in {path_label}")
    return InferenceConfig(
        topk_clients=int(i["topk_clients"]),
        infer_stream_batch_rows=int(i["infer_stream_batch_rows"]),
        num_physical_gpus=int(i["num_physical_gpus"]),
        workers_per_gpu=int(i["workers_per_gpu"]),
        ranking_output=str(i["ranking_output"]),
        rank_user_batch=int(i.get("rank_user_batch", 4096)),
        client_chunk=int(i.get("client_chunk", 8192)),
        use_amp=bool(i.get("use_amp", True)),
        amp_dtype=str(i.get("amp_dtype", "float16")),
        output_min_rows_per_part=int(i.get("output_min_rows_per_part", 100_000)),
        output_parquet_compression=str(i.get("output_parquet_compression", "zstd")),
        debug_cuda=bool(i.get("debug_cuda", False)),
        user_tower_backend=str(i.get("user_tower_backend", "pytorch")),
        user_tower_onnx_uri=str(i["user_tower_onnx_uri"]) if i.get("user_tower_onnx_uri") else None,
        trt_fp16_enable=bool(i.get("trt_fp16_enable", True)),
        trt_engine_cache_enable=bool(i.get("trt_engine_cache_enable", True)),
        trt_engine_cache_path=str(i["trt_engine_cache_path"]) if i.get("trt_engine_cache_path") else None,
        max_files=int(i["max_files"]) if i.get("max_files") not in (None, "", 0) else None,
        max_users_per_file=int(i["max_users_per_file"])
        if i.get("max_users_per_file") not in (None, "", 0)
        else None,
    )


def load_infer_job_config(path: str | Path) -> InferJobConfig:
    """Load ``configs/infer.yaml`` (standalone inference)."""
    path = Path(path)
    raw: dict[str, Any] = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    if not isinstance(raw, dict):
        raise ValueError(f"Config root must be a mapping, got {type(raw)}")
    p = raw.get("paths") or {}
    for k in ("infer", "artifacts_base"):
        if k not in p or p[k] is None:
            raise KeyError(f"paths.{k} is required in {path}")
    paths = InferPaths(infer=str(p["infer"]), artifacts_base=str(p["artifacts_base"]))
    infer = _parse_inference_section(raw.get("infer") or {}, path_label=str(path))
    return InferJobConfig(paths=paths, infer=infer)


def load_pipeline_config(path: str | Path) -> PipelineConfig:
    """Load full training pipeline config from YAML (see ``configs/train.yaml``)."""
    path = Path(path)
    raw: dict[str, Any] = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    if not isinstance(raw, dict):
        raise ValueError(f"Config root must be a mapping, got {type(raw)}")

    p = raw.get("paths") or {}
    for k in ("train", "val", "infer", "artifacts_base"):
        if k not in p or p[k] is None:
            raise KeyError(f"paths.{k} is required in {path}")
    paths = DataPaths(
        train=str(p["train"]),
        val=str(p["val"]),
        infer=str(p["infer"]),
        artifacts_base=str(p["artifacts_base"]),
    )

    f = raw.get("features") or {}
    for k in ("label_col", "device_id_col", "client_id_col"):
        if k not in f or f[k] is None:
            raise KeyError(f"features.{k} is required in {path}")
    features = FeatureConfig(
        label_col=str(f["label_col"]),
        device_id_col=str(f["device_id_col"]),
        client_id_col=str(f["client_id_col"]),
        user_feature_cols=list(f.get("user_feature_cols") or []),
        client_feature_cols=list(f.get("client_feature_cols") or []),
        user_multi_cols=list(f.get("user_multi_cols") or []),
        client_multi_cols=list(f.get("client_multi_cols") or []),
    )

    t = raw.get("train") or {}
    user_hidden = t.get("user_mlp_hidden")
    if user_hidden is None:
        # Backward-compatible fallback for older configs.
        user_hidden = t.get("mlp_hidden_dims")
    if user_hidden is None:
        raise KeyError(f"train.user_mlp_hidden is required in {path}")
    cmh = t.get("client_mlp_hidden")
    if cmh is None:
        cmh = [256, 256]
    train = TrainConfig(
        experiment_name=str(t.get("experiment_name", "two_tower")),
        run_name=t.get("run_name"),
        seed=int(t.get("seed", 42)),
        batch_size=int(t.get("batch_size", 4096)),
        epochs=int(t.get("epochs", 1)),
        steps_per_epoch=int(t["steps_per_epoch"]) if t.get("steps_per_epoch") not in (None, "", 0) else None,
        lr=float(t.get("lr", 1e-3)),
        weight_decay=float(t.get("weight_decay", 0.0)),
        embed_dim=int(t.get("embed_dim", 1024)),
        user_mlp_hidden=list(user_hidden),
        min_count=int(t.get("min_count", 5)),
        num_oov_buckets=int(t.get("num_oov_buckets", 1000)),
        multi_max_tokens=int(t.get("multi_max_tokens", 32)),
        num_workers=int(t.get("num_workers", 4)),
        device=str(t.get("device", "cuda")),
        train_eval_max_examples=int(t["train_eval_max_examples"])
        if t.get("train_eval_max_examples") not in (None, "", 0)
        else None,
        train_client_eval_max_examples=int(t["train_client_eval_max_examples"])
        if t.get("train_client_eval_max_examples") not in (None, "", 0)
        else None,
        dataloader_pin_memory=bool(t.get("dataloader_pin_memory", True)),
        dataloader_persistent_workers=bool(t.get("dataloader_persistent_workers", True)),
        dataloader_prefetch_factor=int(t.get("dataloader_prefetch_factor", 2)),
        torch_compile=bool(t.get("torch_compile", False)),
        torch_compile_mode=str(t.get("torch_compile_mode", "reduce-overhead")),
        downsample_train=bool(t.get("downsample_train", False)),
        downsample_val=bool(t.get("downsample_val", False)),
        downsample_neg_per_pos=int(t["downsample_neg_per_pos"])
        if t.get("downsample_neg_per_pos") not in (None, "", 0)
        else None,
        downsample_equalize_client_rows=bool(t.get("downsample_equalize_client_rows", True)),
        downsample_random_state=int(t.get("downsample_random_state", 42)),
        batch_balance=bool(t.get("batch_balance", False)),
        batch_balance_neg_per_pos=int(t.get("batch_balance_neg_per_pos", 3)),
        pretrained_emb_dim=int(t.get("pretrained_emb_dim", 128)),
        pretrained_cat_emb_dim=int(t.get("pretrained_cat_emb_dim", 64)),
        freeze_pretrained_base=bool(t.get("freeze_pretrained_base", True)),
        multi_cat_pool=str(t.get("multi_cat_pool", "mean")),
        client_mlp_hidden=list(cmh),
    )

    infer = _parse_inference_section(raw.get("infer") or {}, path_label=str(path))

    d = raw.get("data") or {}
    row_filter = d.get("single_client_row_filter")
    if row_filter is not None and not isinstance(row_filter, dict):
        raise TypeError("data.single_client_row_filter must be a mapping or null")
    hardcoded = d.get("single_client_features_hardcoded")
    if hardcoded is not None and not isinstance(hardcoded, dict):
        raise TypeError("data.single_client_features_hardcoded must be a mapping or null")

    data_load = DataLoadConfig(
        merge_client_metadata=bool(d.get("merge_client_metadata", False)),
        client_metadata_uri=d.get("client_metadata_uri"),
        inject_single_client_metadata=bool(d.get("inject_single_client_metadata", False)),
        single_client_metadata_uri=d.get("single_client_metadata_uri"),
        single_client_row_filter=dict(row_filter) if row_filter else None,
        single_client_features_hardcoded=dict(hardcoded) if hardcoded else None,
    )

    extra = raw.get("extra")
    if extra is not None and not isinstance(extra, dict):
        raise TypeError("extra must be a mapping or null")

    return PipelineConfig(
        paths=paths,
        features=features,
        train=train,
        infer=infer,
        data_load=data_load,
        mlflow_tracking_uri=raw.get("mlflow_tracking_uri"),
        extra=dict(extra) if extra else None,
    )

# ======================================================================================
# URI helpers (S3/local read)
# Source: io/uris.py
# ======================================================================================

from pathlib import Path


def read_uri_bytes(uri: str) -> bytes:
    uri = str(uri)
    if uri.startswith("s3://"):
        import s3fs

        with s3fs.S3FileSystem().open(uri, "rb") as f:
            return f.read()
    return Path(uri).expanduser().resolve().read_bytes()

# ======================================================================================
# Artifact path helpers
# Source: io/paths.py
# ======================================================================================

from pathlib import Path


def artifact_uri(base: str, *relative_parts: str) -> str:
    """Join S3 or local base with relative path parts."""
    rel = "/".join(relative_parts)
    b = base.rstrip("/")
    if b.startswith("s3://"):
        return f"{b}/{rel}"
    return str(Path(b) / rel)

# ======================================================================================
# RunLog (stdout + file)
# Source: io/runlog.py
# ======================================================================================

import os
import socket
import time
from dataclasses import dataclass
from pathlib import Path


@dataclass
class RunLog:
    path: Path

    def write(self, msg: str) -> None:
        ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        line = f"{ts} | {msg}\n"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a", encoding="utf-8") as f:
            f.write(line)
            f.flush()


def _default_logs_dir() -> Path:
    # Use cwd so notebooks + CLI runs land in the same place.
    return Path(os.getcwd()).resolve() / "logs"


def start_run_log(*, kind: str, name: str | None = None, logs_dir: str | Path | None = None) -> RunLog:
    d = Path(logs_dir).resolve() if logs_dir is not None else _default_logs_dir()
    safe = (name or "run").replace(" ", "_")
    stamp = time.strftime("%Y%m%d_%H%M%S", time.localtime())
    p = d / f"{kind}_{safe}_{stamp}.log"
    rl = RunLog(path=p)
    rl.write(f"START kind={kind} name={name or ''} host={socket.gethostname()} pid={os.getpid()}")
    return rl

# ======================================================================================
# Vocabulary helpers
# Source: features/vocab.py
# ======================================================================================

import hashlib
import re
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class CatVocab:
    token_to_id: dict[str, int]
    n_frequent: int
    num_oov_buckets: int

    @property
    def size(self) -> int:
        # 0 reserved for missing/padding
        return int(self.n_frequent + self.num_oov_buckets + 1)

    def encode_scalar(self, raw) -> int:
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            return 0
        s = str(raw).strip()
        if s == "" or s.lower() == "nan" or s == "__NA__":
            return 0
        tid = self.token_to_id.get(s)
        if tid is not None:
            return int(tid)
        h = int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)
        return int(self.n_frequent + 1 + (h % self.num_oov_buckets))


def build_cat_vocab(series, min_count: int, num_oov_buckets: int) -> CatVocab:
    s = series.fillna("__NA__").astype(str)
    vc = s.value_counts()
    frequent = vc[vc > int(min_count)]
    tokens = list(frequent.index)
    token_to_id = {tok: i + 1 for i, tok in enumerate(tokens)}
    return CatVocab(token_to_id=token_to_id, n_frequent=len(tokens), num_oov_buckets=int(num_oov_buckets))


def parse_multi_cell(val) -> list[str]:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, tuple)):
        return [str(x).strip() for x in val if str(x).strip() and str(x).strip().lower() != "nan"]
    if isinstance(val, np.ndarray):
        return [
            str(x).strip()
            for x in val.flatten().tolist()
            if str(x).strip() and str(x).strip().lower() != "nan"
        ]
    s = str(val).strip()
    if not s or s == "__NA__" or s.lower() == "nan":
        return []
    parts = re.split(r"[\|,;]+|\s+", s)
    return [p for p in parts if p]


def build_multi_token_vocab(series, min_count: int, num_oov_buckets: int) -> CatVocab:
    ctr: Counter[str] = Counter()
    for v in series:
        for tok in parse_multi_cell(v):
            ctr[tok] += 1
    tokens = sorted([t for t, n in ctr.items() if n > int(min_count)], key=lambda t: (-ctr[t], t))
    token_to_id = {tok: i + 1 for i, tok in enumerate(tokens)}
    return CatVocab(token_to_id=token_to_id, n_frequent=len(tokens), num_oov_buckets=int(num_oov_buckets))


def vocab_to_dict(v: CatVocab) -> dict:
    return {"token_to_id": dict(v.token_to_id), "n_frequent": int(v.n_frequent), "num_oov_buckets": int(v.num_oov_buckets)}


def vocab_from_dict(d: dict) -> CatVocab:
    return CatVocab(token_to_id=dict(d["token_to_id"]), n_frequent=int(d["n_frequent"]), num_oov_buckets=int(d["num_oov_buckets"]))

# ======================================================================================
# User tower model architecture
# Source: model/two_tower.py
# ======================================================================================

import math
from typing import List

import torch
import torch.nn as nn


def embedding_dim_for_cardinality(vocab_size: int) -> int:
    v = int(max(1, vocab_size))
    if v < 100:
        return 6
    if v < 10_000:
        return 20
    if v < 1_000_000:
        return 64
    return 128


class CatEmbedder(nn.Module):
    """Categorical embedder: random init or hash-pretrained + linear projection."""

    def __init__(
        self,
        vocab_sizes: List[int],
        use_pretrained: bool = False,
        pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_emb_dim_per_col: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        self.use_pretrained = use_pretrained

        if use_pretrained:
            self.emb_dims = [target_emb_dim_per_col] * len(vocab_sizes)
            self.embs = nn.ModuleList([nn.Embedding(int(v), pretrained_emb_dim, padding_idx=0) for v in vocab_sizes])
            if pretrained_weights is not None:
                for emb, w_np in zip(self.embs, pretrained_weights):
                    w = torch.from_numpy(w_np).float()
                    n = min(w.shape[0], emb.weight.shape[0])
                    emb.weight.data[:n].copy_(w[:n])
            if freeze_base:
                for emb in self.embs:
                    emb.weight.requires_grad_(False)
            self.projs: nn.ModuleList | None = nn.ModuleList(
                [nn.Linear(pretrained_emb_dim, target_emb_dim_per_col, bias=False) for _ in vocab_sizes]
            )
        else:
            self.emb_dims = [embedding_dim_for_cardinality(v) for v in vocab_sizes]
            self.embs = nn.ModuleList([nn.Embedding(int(v), int(d)) for v, d in zip(vocab_sizes, self.emb_dims)])
            self.projs = None

    def output_dim(self) -> int:
        return int(sum(self.emb_dims))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.numel() == 0:
            return torch.zeros((x.shape[0], 0), device=x.device)
        if self.use_pretrained and self.projs is not None:
            parts = [proj(emb(x[:, i])) for i, (emb, proj) in enumerate(zip(self.embs, self.projs))]
        else:
            parts = [emb(x[:, i]) for i, emb in enumerate(self.embs)]
        return torch.cat(parts, dim=1)


class TokenSeqAttentionPool(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.score = nn.Linear(int(d), 1)

    def forward(self, e: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        s = self.score(e).squeeze(-1)
        s = s.masked_fill(mask <= 0, -1e9)
        w = torch.softmax(s, dim=1).unsqueeze(-1)
        return (e * w).sum(dim=1)


class PooledMultiCatEmbedder(nn.Module):
    def __init__(self, vocab_sizes: List[int], emb_dims: List[int], pool: str = "mean"):
        super().__init__()
        self.pool = pool
        self.emb_dims = [int(d) for d in emb_dims]
        self.embs = nn.ModuleList(
            [nn.Embedding(int(v), int(d), padding_idx=0) for v, d in zip(vocab_sizes, self.emb_dims)]
        )
        self._attn: nn.ModuleList | None = None
        if pool == "attention":
            self._attn = nn.ModuleList([TokenSeqAttentionPool(d) for d in self.emb_dims])

    def output_dim(self) -> int:
        return int(sum(self.emb_dims))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] == 0:
            return torch.zeros((x.shape[0], 0), device=x.device)
        outs = []
        for i, emb in enumerate(self.embs):
            ids = x[:, i, :]
            e = emb(ids)
            mask = (ids != 0).float()
            if self.pool == "sum":
                pooled = (e * mask.unsqueeze(-1)).sum(dim=1)
            elif self.pool == "attention" and self._attn is not None:
                pooled = self._attn[i](e, mask)
            else:
                denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
                pooled = (e * mask.unsqueeze(-1)).sum(dim=1) / denom
            outs.append(pooled)
        return torch.cat(outs, dim=1)


def _safe_l2_normalize(x: torch.Tensor, dim: int = 1, eps: float = 1e-6) -> torch.Tensor:
    denom = x.norm(p=2, dim=dim, keepdim=True).clamp_min(eps)
    return x / denom


class DCNv2CrossLayer(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(input_dim, input_dim) * 0.01)
        self.bias = nn.Parameter(torch.zeros(input_dim))

    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        cross = xl @ self.weight
        cross = cross + self.bias
        cross = x0 * cross
        return cross + xl


class DCNv2UserTower(nn.Module):
    def __init__(
        self,
        user_vocab_sizes: List[int],
        user_num_dim: int,
        emb_dim: int = 1024,
        num_cross_layers: int = 3,
        deep_hidden: List[int] | None = None,
        user_multi_vocab_sizes: List[int] | None = None,
        user_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        user_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        if deep_hidden is None:
            deep_hidden = [512, 512]

        self.user_cat = CatEmbedder(
            user_vocab_sizes,
            use_pretrained=use_pretrained_cat,
            pretrained_weights=user_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_emb_dim_per_col=target_cat_emb_dim,
            freeze_base=freeze_base,
        )
        ms = list(user_multi_vocab_sizes or [])
        if ms:
            ed = user_multi_emb_dims or [embedding_dim_for_cardinality(v) for v in ms]
            self.user_multi = PooledMultiCatEmbedder(ms, ed, pool=multi_pool)
        else:
            self.user_multi = None

        cat_w = self.user_cat.output_dim() + (self.user_multi.output_dim() if self.user_multi else 0)
        input_dim = int(cat_w + user_num_dim)

        self.cross_layers = nn.ModuleList([DCNv2CrossLayer(input_dim) for _ in range(num_cross_layers)])

        deep_layers: list[nn.Module] = []
        prev = input_dim
        for h in deep_hidden:
            deep_layers.append(nn.Linear(prev, h))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.BatchNorm1d(h))
            prev = h
        self.deep = nn.Sequential(*deep_layers)

        self.fc = nn.Linear(prev + input_dim, emb_dim)

    def forward(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        cat = self.user_cat(user_cat)
        if self.user_multi is not None:
            cat = torch.cat([cat, self.user_multi(user_multi)], dim=1)
        x0 = torch.cat([cat, user_num], dim=1)
        xl = x0
        for layer in self.cross_layers:
            xl = layer(x0, xl)
        cross_out = xl
        deep_out = self.deep(x0)
        emb = self.fc(torch.cat([cross_out, deep_out], dim=1))
        return _safe_l2_normalize(emb, dim=1)


class UserMLPTower(nn.Module):
    def __init__(
        self,
        user_vocab_sizes: List[int],
        user_num_dim: int,
        emb_dim: int = 1024,
        hidden: List[int] | None = None,
        user_multi_vocab_sizes: List[int] | None = None,
        user_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        user_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        if hidden is None:
            hidden = [512, 256]

        self.user_cat = CatEmbedder(
            user_vocab_sizes,
            use_pretrained=use_pretrained_cat,
            pretrained_weights=user_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_emb_dim_per_col=target_cat_emb_dim,
            freeze_base=freeze_base,
        )
        ms = list(user_multi_vocab_sizes or [])
        if ms:
            ed = user_multi_emb_dims or [embedding_dim_for_cardinality(v) for v in ms]
            self.user_multi = PooledMultiCatEmbedder(ms, ed, pool=multi_pool)
        else:
            self.user_multi = None

        cat_w = self.user_cat.output_dim() + (self.user_multi.output_dim() if self.user_multi else 0)
        input_dim = int(cat_w + user_num_dim)

        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(h))
            prev = h
        layers.append(nn.Linear(prev, emb_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        cat = self.user_cat(user_cat)
        if self.user_multi is not None:
            cat = torch.cat([cat, self.user_multi(user_multi)], dim=1)
        x = torch.cat([cat, user_num], dim=1)
        emb = self.net(x)
        return _safe_l2_normalize(emb, dim=1)


class ClientMLPTower(nn.Module):
    def __init__(
        self,
        client_vocab_sizes: List[int],
        client_num_dim: int,
        emb_dim: int = 1024,
        hidden: List[int] | None = None,
        client_multi_vocab_sizes: List[int] | None = None,
        client_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        client_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        if hidden is None:
            hidden = [256, 256]

        self.client_cat = CatEmbedder(
            client_vocab_sizes,
            use_pretrained=use_pretrained_cat,
            pretrained_weights=client_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_emb_dim_per_col=target_cat_emb_dim,
            freeze_base=freeze_base,
        )
        ms = list(client_multi_vocab_sizes or [])
        if ms:
            ed = client_multi_emb_dims or [embedding_dim_for_cardinality(v) for v in ms]
            self.client_multi = PooledMultiCatEmbedder(ms, ed, pool=multi_pool)
        else:
            self.client_multi = None

        cat_w = self.client_cat.output_dim() + (self.client_multi.output_dim() if self.client_multi else 0)
        input_dim = int(cat_w + client_num_dim)

        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(h))
            prev = h
        layers.append(nn.Linear(prev, emb_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, client_cat: torch.Tensor, client_num: torch.Tensor, client_multi: torch.Tensor) -> torch.Tensor:
        cat = self.client_cat(client_cat)
        if self.client_multi is not None:
            cat = torch.cat([cat, self.client_multi(client_multi)], dim=1)
        x = torch.cat([cat, client_num], dim=1)
        emb = self.net(x)
        return _safe_l2_normalize(emb, dim=1)


class TwoTowerModel(nn.Module):
    def __init__(
        self,
        user_vocab_sizes: List[int],
        user_num_dim: int,
        client_vocab_sizes: List[int],
        client_num_dim: int,
        emb_dim: int = 1024,
        user_multi_vocab_sizes: List[int] | None = None,
        user_multi_emb_dims: List[int] | None = None,
        client_multi_vocab_sizes: List[int] | None = None,
        client_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        user_cat_pretrained_weights: list | None = None,
        client_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_pretrained_base: bool = False,
        user_hidden: List[int] | None = None,
        client_hidden: List[int] | None = None,
    ):
        super().__init__()
        self.log_scale = nn.Parameter(torch.tensor(math.log(20.0)))
        self.user_tower = UserMLPTower(
            user_vocab_sizes,
            user_num_dim,
            emb_dim=emb_dim,
            hidden=user_hidden,
            user_multi_vocab_sizes=user_multi_vocab_sizes,
            user_multi_emb_dims=user_multi_emb_dims,
            multi_pool=multi_pool,
            use_pretrained_cat=use_pretrained_cat,
            user_cat_pretrained_weights=user_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_cat_emb_dim=target_cat_emb_dim,
            freeze_base=freeze_pretrained_base,
        )
        self.client_tower = ClientMLPTower(
            client_vocab_sizes,
            client_num_dim,
            emb_dim=emb_dim,
            hidden=client_hidden,
            client_multi_vocab_sizes=client_multi_vocab_sizes,
            client_multi_emb_dims=client_multi_emb_dims,
            multi_pool=multi_pool,
            use_pretrained_cat=use_pretrained_cat,
            client_cat_pretrained_weights=client_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_cat_emb_dim=target_cat_emb_dim,
            freeze_base=freeze_pretrained_base,
        )

    def forward(
        self,
        user_cat: torch.Tensor,
        user_num: torch.Tensor,
        client_cat: torch.Tensor,
        client_num: torch.Tensor,
        user_multi: torch.Tensor,
        client_multi: torch.Tensor,
    ):
        user_emb = self.user_tower(user_cat, user_num, user_multi)
        client_emb = self.client_tower(client_cat, client_num, client_multi)
        scale = self.log_scale.clamp(math.log(1.0), math.log(100.0)).exp()
        logits = (user_emb * client_emb).sum(dim=1, keepdim=True) * scale
        return logits, user_emb, client_emb


def build_two_tower_model(fa, cfg) -> TwoTowerModel:
    """Construct model from feature artifacts + pipeline config."""

    assert isinstance(cfg, PipelineConfig)
    assert isinstance(fa, FeatureArtifacts)

    tc = cfg.train
    user_vocab_sizes = [fa.user_vocabs[c].size for c in fa.user_cat_cols]
    client_vocab_sizes = [fa.client_vocabs[c].size for c in fa.client_cat_cols]
    user_multi_vocab_sizes = [fa.user_multi_vocabs[c].size for c in fa.user_multi_cols]
    client_multi_vocab_sizes = [fa.client_multi_vocabs[c].size for c in fa.client_multi_cols]
    user_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in user_multi_vocab_sizes]
    client_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in client_multi_vocab_sizes]

    user_num_dim = len(fa.user_num_cols)
    client_num_dim = len(fa.client_num_cols)

    use_pt = bool(fa.user_cat_pretrained_weights) and bool(fa.client_cat_pretrained_weights)
    user_w = [fa.user_cat_pretrained_weights[c] for c in fa.user_cat_cols] if use_pt and fa.user_cat_cols else None
    client_w = (
        [fa.client_cat_pretrained_weights[c] for c in fa.client_cat_cols] if use_pt and fa.client_cat_cols else None
    )

    return TwoTowerModel(
        user_vocab_sizes=user_vocab_sizes,
        user_num_dim=user_num_dim,
        client_vocab_sizes=client_vocab_sizes,
        client_num_dim=client_num_dim,
        emb_dim=tc.embed_dim,
        user_hidden=list(tc.user_mlp_hidden),
        client_hidden=list(tc.client_mlp_hidden),
        user_multi_vocab_sizes=user_multi_vocab_sizes or None,
        user_multi_emb_dims=user_multi_emb_dims or None,
        client_multi_vocab_sizes=client_multi_vocab_sizes or None,
        client_multi_emb_dims=client_multi_emb_dims or None,
        multi_pool=tc.multi_cat_pool,
        use_pretrained_cat=use_pt,
        user_cat_pretrained_weights=user_w,
        client_cat_pretrained_weights=client_w,
        pretrained_emb_dim=tc.pretrained_emb_dim,
        target_cat_emb_dim=tc.pretrained_cat_emb_dim,
        freeze_pretrained_base=tc.freeze_pretrained_base,
    )

# ======================================================================================
# User tower runtime backends
# Source: inference/user_tower_backend.py
# ======================================================================================

from dataclasses import dataclass
from typing import Protocol

import numpy as np
import torch
from torch import nn


class UserTowerBackend(Protocol):
    """Runtime backend abstraction for user tower embedding inference."""

    backend_name: str

    def infer(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        ...


@dataclass
class TorchUserTowerBackend:
    """PyTorch backend (current default path)."""

    model: nn.Module
    backend_name: str = "pytorch"

    def infer(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        return self.model(user_cat, user_num, user_multi)


@dataclass
class OnnxRuntimeTensorRTUserTowerBackend:
    """
    TensorRT backend via ONNX Runtime provider stack.

    Provider order:
    1) TensorRTExecutionProvider
    2) CUDAExecutionProvider
    3) CPUExecutionProvider
    """

    session: object
    output_name: str
    device: torch.device
    input_names: tuple[str, ...]
    backend_name: str = "tensorrt_onnxruntime"

    def infer(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        # ORT expects CPU numpy inputs. Feed only names present in the exported graph.
        cat_np = user_cat.detach().cpu().numpy().astype(np.int64, copy=False)
        num_np = user_num.detach().cpu().numpy().astype(np.float32, copy=False)
        multi_np = user_multi.detach().cpu().numpy().astype(np.int64, copy=False)
        inp: dict[str, np.ndarray] = {}
        for name in self.input_names:
            if name == "user_cat":
                inp[name] = cat_np
            elif name == "user_num":
                inp[name] = num_np
            elif name == "user_multi":
                inp[name] = multi_np
        if not inp:
            raise RuntimeError(
                f"ONNX session has unsupported input names: {self.input_names}. "
                "Expected one or more of: user_cat, user_num, user_multi."
            )
        out = self.session.run([self.output_name], inp)[0]
        return torch.from_numpy(np.asarray(out, dtype=np.float32)).to(self.device, non_blocking=True)

# ======================================================================================
# Parquet input listing
# Source: inference/list_inputs.py
# ======================================================================================

from pathlib import Path


def list_parquet_inputs(infer_path: str) -> list[str]:
    """Return parquet file URIs/paths under ``infer_path`` (S3 prefix, local dir, or single file)."""
    infer_path = str(infer_path).strip()
    if infer_path.endswith(".parquet"):
        p = Path(infer_path)
        if not infer_path.startswith("s3://"):
            if not p.is_file():
                raise FileNotFoundError(f"Parquet file not found: {infer_path!r}")
            return [str(p.resolve())]
        return [infer_path]

    if infer_path.startswith("s3://"):
        import s3fs

        fs = s3fs.S3FileSystem()
        no_scheme = infer_path.replace("s3://", "").rstrip("/")
        if not fs.exists(no_scheme):
            raise FileNotFoundError(f"S3 path does not exist: {infer_path!r}")
        found = fs.find(no_scheme)

        def _is_leaf_parquet(key: str) -> bool:
            base = key.rsplit("/", 1)[-1]
            if not base or base.startswith("_") or base.startswith("."):
                return False
            return base.endswith(".parquet") or "parquet" in base or base.startswith("part-")

        file_uris = sorted({f"s3://{k}" for k in found if _is_leaf_parquet(k)})
        if file_uris:
            return file_uris
        top = fs.ls(no_scheme)
        print(f"[list_parquet_inputs] No leaf parquet; sharding by {len(top)} top-level prefix(es)")
        return sorted({f"s3://{k}" for k in top})

    root = Path(infer_path).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"Path does not exist: {infer_path!r}")
    if root.is_file():
        return [str(root)]
    files = sorted(str(p) for p in root.rglob("*.parquet") if p.is_file())
    if not files:
        raise FileNotFoundError(f"No .parquet files under {infer_path!r}")
    return files

# ======================================================================================
# Training artifact URIs + vocab loader
# Source: inference/artifact_paths.py
# ======================================================================================

import pickle



def training_artifact_uris(artifacts_base: str) -> dict[str, str]:
    base = artifacts_base.rstrip("/")
    return {
        "user_tower": artifact_uri(base, "artifacts", "user_tower", "user_tower_state.pt"),
        "vocab": artifact_uri(base, "artifacts", "vocab_artifact", "vocab_artifact.pkl"),
        "client_embeddings": artifact_uri(base, "artifacts", "client_embeddings", "client_embeddings.parquet"),
    }


def load_vocab_artifact_pickle(uri: str) -> dict:
    return pickle.loads(read_uri_bytes(uri))

# ======================================================================================
# Inference worker + ranking core
# Source: inference/worker.py
# ======================================================================================

import gc
import hashlib
import io
import os
import tempfile
import time
import traceback
from contextlib import nullcontext
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import pyarrow.dataset as pads
import torch
from torch import nn



def _iter_record_batches(dset: Any, cols: list[str], batch_size: int):
    try:
        yield from pads.Scanner.from_dataset(dset, columns=cols, batch_size=batch_size).to_batches()
        return
    except Exception:
        pass
    for frag in dset.get_fragments():
        yield from frag.to_batches(batch_size=batch_size, columns=cols)


def _prepare_frame(
    pdf: pl.DataFrame,
    user_cat_cols: list[str],
    user_num_cols: list[str],
    user_multi_cols: list[str],
) -> pl.DataFrame:
    add_cols: list[pl.Expr] = []
    for c in user_cat_cols:
        if c not in pdf.schema:
            add_cols.append(pl.lit("__NA__").alias(c))
    for c in user_num_cols:
        if c not in pdf.schema:
            add_cols.append(pl.lit(0.0).alias(c))
    for c in user_multi_cols:
        if c not in pdf.schema:
            add_cols.append(pl.lit(None).alias(c))
    if add_cols:
        pdf = pdf.with_columns(add_cols)
    return pdf


def _encode_cats_pl(df: pl.DataFrame, cat_cols: list[str], vocabs: dict) -> torch.Tensor:
    if not cat_cols:
        return torch.zeros((len(df), 0), dtype=torch.long)
    n = len(df)
    out = np.empty((n, len(cat_cols)), dtype=np.int64)
    for j, c in enumerate(cat_cols):
        vocab = vocabs[c]
        vals = df[c].to_list() if c in df.schema else ["__NA__"] * n
        out[:, j] = np.fromiter((vocab.encode_scalar(v) for v in vals), dtype=np.int64, count=n)
    return torch.from_numpy(out)


def _encode_nums_pl(df: pl.DataFrame, num_cols: list[str]) -> torch.Tensor:
    if not num_cols:
        return torch.zeros((len(df), 0), dtype=torch.float32)
    cols: list[np.ndarray] = []
    n = len(df)
    for c in num_cols:
        if c in df.schema:
            s = df[c].cast(pl.Float32, strict=False).fill_null(0.0)
            arr = s.to_numpy()
        else:
            arr = np.zeros((n,), dtype=np.float32)
        cols.append(np.asarray(arr, dtype=np.float32))
    x = np.stack(cols, axis=1)
    return torch.from_numpy(x)


def _encode_multi_matrix_pl(
    df: pl.DataFrame, cols: list[str], vocabs: dict, max_tokens: int
) -> torch.Tensor:
    """Shape (N, len(cols), max_tokens); 0 = pad / empty slot."""
    n = len(df)
    f = len(cols)
    mt = int(max_tokens)
    if f == 0:
        return torch.zeros((n, 0, max(1, mt)), dtype=torch.long)
    out = np.zeros((n, f, mt), dtype=np.int64)
    for j, c in enumerate(cols):
        vcb = vocabs[c]
        vals = df[c].to_list() if c in df.schema else [None] * n
        for i, val in enumerate(vals):
            toks = parse_multi_cell(val)[:mt]
            for k, tok in enumerate(toks):
                tid = vcb.token_to_id.get(tok)
                if tid is not None:
                    out[i, j, k] = tid
                else:
                    h = int(hashlib.md5(tok.encode("utf-8")).hexdigest(), 16)
                    out[i, j, k] = int(vcb.n_frequent + 1 + (h % vcb.num_oov_buckets))
    return torch.from_numpy(out)


def _rank_batch_topk(
    *,
    infer_df: pl.DataFrame,
    device_id_col: str,
    user_cat_cols: list[str],
    user_num_cols: list[str],
    user_multi_cols: list[str],
    user_vocabs: dict,
    user_multi_vocabs: dict,
    multi_max_tokens: int,
    expected_vocab_sizes: list[int],
    expected_cat_dim: int,
    expected_num_dim: int,
    user_tower_backend: UserTowerBackend,
    client_emb_t: torch.Tensor,
    client_ids_np: np.ndarray,
    topk: int,
    client_chunk: int,
    use_amp: bool,
    amp_dtype_torch: torch.dtype,
    to_device: torch.device,
    worker_id: int,
    forward_probe_state: dict[str, int] | None = None,
) -> pl.DataFrame:
    device_ids = infer_df[device_id_col].to_numpy()
    uc = _encode_cats_pl(infer_df, user_cat_cols, user_vocabs)
    un = _encode_nums_pl(infer_df, user_num_cols)
    um = _encode_multi_matrix_pl(infer_df, user_multi_cols, user_multi_vocabs, multi_max_tokens)

    if uc.shape[1] < expected_cat_dim:
        uc = torch.cat([uc, torch.zeros((uc.shape[0], expected_cat_dim - uc.shape[1]), dtype=uc.dtype)], dim=1)
    elif uc.shape[1] > expected_cat_dim:
        uc = uc[:, :expected_cat_dim]
    if un.shape[1] < expected_num_dim:
        un = torch.cat([un, torch.zeros((un.shape[0], expected_num_dim - un.shape[1]), dtype=un.dtype)], dim=1)
    elif un.shape[1] > expected_num_dim:
        un = un[:, :expected_num_dim]
    for i, vmax in enumerate(expected_vocab_sizes):
        if i < uc.shape[1]:
            uc[:, i] = uc[:, i].clamp(0, int(vmax) - 1)

    uc = uc.to(to_device)
    un = un.to(to_device)
    um = um.to(to_device)

    bsz = len(infer_df)
    n_clients = int(client_emb_t.shape[0])

    use_torch_amp = user_tower_backend.backend_name == "pytorch"
    ctx = (
        torch.autocast(device_type="cuda", dtype=amp_dtype_torch)
        if use_torch_amp and to_device.type == "cuda" and use_amp
        else nullcontext()
    )

    with torch.inference_mode(), ctx:
        # Baseline latency probe: log single-row user-tower forward for first few records only.
        if forward_probe_state is not None and int(forward_probe_state.get("remaining", 0)) > 0 and bsz > 0:
            probe_n = min(int(forward_probe_state.get("remaining", 0)), bsz)
            for i in range(probe_n):
                t0 = time.perf_counter()
                _ = user_tower_backend.infer(uc[i : i + 1], un[i : i + 1], um[i : i + 1])
                if to_device.type == "cuda":
                    torch.cuda.synchronize()
                elapsed_ms = (time.perf_counter() - t0) * 1000.0
                print(
                    f"[infer][worker={worker_id}] single_forward_ms={elapsed_ms:.3f} device_id={device_ids[i]}",
                    flush=True,
                )
            forward_probe_state["remaining"] = int(forward_probe_state.get("remaining", 0)) - probe_n

        uemb = user_tower_backend.infer(uc, un, um)

        best_scores = torch.full((bsz, topk), -1.0, device=to_device, dtype=torch.float32)
        best_idx = torch.full((bsz, topk), -1, device=to_device, dtype=torch.int64)

        for start in range(0, n_clients, client_chunk):
            end = min(start + client_chunk, n_clients)
            chunk = client_emb_t[start:end]
            scores = (uemb @ chunk.T).float()
            k = min(topk, end - start)
            cs, ci = torch.topk(scores, k=k, dim=1)
            ci = ci + start
            ms = torch.cat([best_scores, cs], dim=1)
            mi = torch.cat([best_idx, ci], dim=1)
            best_scores, sel = torch.topk(ms, k=topk, dim=1)
            best_idx = torch.gather(mi, 1, sel)
            del chunk, scores, cs, ci, ms, mi, sel

    s_np = best_scores.detach().cpu().numpy().astype(np.float32)
    i_np = best_idx.detach().cpu().numpy().astype(np.int64)

    return pl.DataFrame(
        {
            device_id_col: np.repeat(device_ids, topk),
            "client_id": client_ids_np[i_np.reshape(-1)],
            "score": s_np.reshape(-1),
            "rank": np.tile(np.arange(1, topk + 1, dtype=np.int32), bsz),
        }
    )


def _materialize_uri_to_local(uri: str, suffix: str) -> str:
    """Download/load URI bytes to a local temp file path."""
    if uri.startswith("s3://"):
        raw = read_uri_bytes(uri)
        fd, p = tempfile.mkstemp(suffix=suffix, prefix="two_tower_")
        os.close(fd)
        Path(p).write_bytes(raw)
        return p
    return str(Path(uri))


def _build_user_tower_backend(
    *,
    backend_name: str,
    onnx_uri: str | None,
    to_device: torch.device,
    use_amp: bool,
    amp_dtype_torch: torch.dtype,
    trt_fp16_enable: bool,
    trt_engine_cache_enable: bool,
    trt_engine_cache_path: str | None,
    expected_vocab_sizes: list[int],
    expected_num_dim: int,
    emb_dim: int,
    um: list[int],
    umd: list[int],
    mp: str,
    use_pt_cat: bool,
    hidden: list[int],
    state: dict[str, Any],
) -> tuple[UserTowerBackend, nn.Module | None]:
    backend_name = str(backend_name or "pytorch").lower()
    if backend_name == "pytorch":
        common_kwargs = dict(
            user_vocab_sizes=expected_vocab_sizes,
            user_num_dim=expected_num_dim,
            emb_dim=emb_dim,
            user_multi_vocab_sizes=um if um else None,
            user_multi_emb_dims=umd if umd else None,
            multi_pool=mp,
            use_pretrained_cat=use_pt_cat,
            pretrained_emb_dim=int(state.get("pretrained_emb_dim", 128)),
            target_cat_emb_dim=int(state.get("target_cat_emb_dim", 64)),
            freeze_base=bool(state.get("freeze_base", True)),
        )
        arch = str(state.get("user_tower_arch", "dcnv2")).lower()
        if arch == "mlp":
            user_tower_model = UserMLPTower(hidden=hidden, **common_kwargs).to(to_device)
        else:
            user_tower_model = DCNv2UserTower(
                num_cross_layers=int(state.get("num_cross_layers", 3)),
                deep_hidden=hidden,
                **common_kwargs,
            ).to(to_device)
        user_tower_model.load_state_dict(state["state_dict"])
        user_tower_model.eval()
        if hasattr(torch, "compile") and to_device.type == "cuda":
            try:
                user_tower_model = torch.compile(user_tower_model, mode="reduce-overhead")
            except Exception:
                pass
        return TorchUserTowerBackend(model=user_tower_model), user_tower_model

    if backend_name == "tensorrt_onnxruntime":
        if onnx_uri is None:
            raise ValueError(
                "infer.user_tower_onnx_uri is required when infer.user_tower_backend='tensorrt_onnxruntime'"
            )
        try:
            import onnxruntime as ort
        except Exception as e:
            raise RuntimeError(
                "onnxruntime is required for tensorrt_onnxruntime backend. Install onnxruntime-gpu."
            ) from e

        onnx_local = _materialize_uri_to_local(onnx_uri, suffix=".onnx")
        trt_cache = trt_engine_cache_path or str(Path(tempfile.gettempdir()) / "two_tower_trt_cache")
        provider_options = [
            {
                "trt_fp16_enable": bool(trt_fp16_enable and use_amp and amp_dtype_torch == torch.float16),
                "trt_engine_cache_enable": bool(trt_engine_cache_enable),
                "trt_engine_cache_path": trt_cache,
            },
            {},
            {},
        ]
        session = ort.InferenceSession(
            onnx_local,
            providers=[
                "TensorrtExecutionProvider",
                "CUDAExecutionProvider",
                "CPUExecutionProvider",
            ],
            provider_options=provider_options,
        )
        out_name = session.get_outputs()[0].name
        input_names = tuple(i.name for i in session.get_inputs())
        print(f"  [backend:tensorrt_onnxruntime] onnx inputs={input_names} output={out_name}", flush=True)
        return OnnxRuntimeTensorRTUserTowerBackend(
            session=session,
            output_name=out_name,
            device=to_device,
            input_names=input_names,
        ), None

    raise ValueError(f"Unsupported infer.user_tower_backend={backend_name!r}")


def tt_infer_worker(
    worker_id: int,
    file_queue: Any,
    status_queue: Any,
    user_tower_uri: str,
    client_embeddings_uri: str,
    infer_ranking_output: str,
    device_id_col: str,
    user_cat_cols: list[str],
    user_num_cols: list[str],
    user_multi_cols: list[str],
    user_vocabs_raw: dict[str, dict],
    user_multi_vocabs_raw: dict[str, dict],
    multi_max_tokens: int,
    rank_user_batch: int,
    topk_clients: int,
    client_chunk: int,
    use_amp: bool,
    amp_dtype_str: str,
    output_min_rows: int,
    output_compression: str,
    infer_stream_batch_rows: int,
    workers_per_gpu: int,
    max_users_per_file: int | None,
    user_tower_backend: str,
    user_tower_onnx_uri: str | None,
    trt_fp16_enable: bool,
    trt_engine_cache_enable: bool,
    trt_engine_cache_path: str | None,
) -> None:
    """Multiprocessing worker: load tower + client matrix once, consume parquet paths from ``file_queue``."""
    os.environ["POLARS_MAX_THREADS"] = "2"
    try:
        pl.Config.set_num_threads(2)
    except Exception:
        pass

    gpu_id = worker_id // max(workers_per_gpu, 1)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

    to_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp_dtype_torch = getattr(torch, amp_dtype_str, torch.float16)

    user_vocabs = {k: vocab_from_dict(v) for k, v in user_vocabs_raw.items()}
    user_multi_vocabs = {k: vocab_from_dict(v) for k, v in user_multi_vocabs_raw.items()}

    t_load = time.time()
    raw = read_uri_bytes(user_tower_uri)
    _buf = io.BytesIO(raw)
    try:
        state = torch.load(_buf, map_location="cpu", weights_only=False)
    except TypeError:
        _buf.seek(0)
        state = torch.load(_buf, map_location="cpu")

    emb_dim = int(state["emb_dim"])
    expected_vocab_sizes = list(state.get("user_vocab_sizes", []))
    expected_cat_dim = len(expected_vocab_sizes)
    expected_num_dim = int(state.get("user_num_dim", 0))
    um = state.get("user_multi_vocab_sizes") or []
    umd = state.get("user_multi_emb_dims") or []
    mp = state.get("multi_cat_pool", "mean")
    use_pt_cat = bool(state.get("use_pretrained_cat", False))
    hidden = state.get("user_hidden")
    if hidden is None:
        hidden = state.get("user_deep_hidden")
    hidden = list(hidden or [512, 512])

    backend, user_tower_model = _build_user_tower_backend(
        backend_name=user_tower_backend,
        onnx_uri=user_tower_onnx_uri,
        to_device=to_device,
        use_amp=use_amp,
        amp_dtype_torch=amp_dtype_torch,
        trt_fp16_enable=trt_fp16_enable,
        trt_engine_cache_enable=trt_engine_cache_enable,
        trt_engine_cache_path=trt_engine_cache_path,
        expected_vocab_sizes=expected_vocab_sizes,
        expected_num_dim=expected_num_dim,
        emb_dim=emb_dim,
        um=list(um),
        umd=list(umd),
        mp=str(mp),
        use_pt_cat=use_pt_cat,
        hidden=hidden,
        state=state,
    )
    print(f"  Worker {worker_id} backend={backend.backend_name}", flush=True)
    del state
    gc.collect()

    client_emb_pl = pl.read_parquet(client_embeddings_uri)
    client_emb_pl = client_emb_pl.unique(subset=["client_id"], keep="first", maintain_order=False)
    client_ids_np = client_emb_pl["client_id"].to_numpy(allow_copy=True)
    emb_list = client_emb_pl["embedding"].to_list()
    del client_emb_pl
    client_emb_np = np.stack([np.asarray(e, dtype=np.float32) for e in emb_list], axis=0)
    del emb_list
    client_emb_t = torch.from_numpy(client_emb_np).to(to_device)
    del client_emb_np
    gc.collect()

    n_clients = int(client_emb_t.shape[0])
    topk = min(int(topk_clients), n_clients)

    print(
        f"  Worker {worker_id} -> GPU {gpu_id} | clients={n_clients} emb_dim={emb_dim} | load={time.time() - t_load:.1f}s",
        flush=True,
    )

    part = 0
    out_buf: list[pl.DataFrame] = []
    out_rows = 0
    out_prefix = infer_ranking_output
    # Log only the first few single-row forward passes to estimate baseline inference latency.
    forward_probe_state: dict[str, int] = {"remaining": 5}

    def _flush(force: bool = False) -> None:
        nonlocal part, out_buf, out_rows
        if not out_buf or (not force and out_rows < output_min_rows):
            return
        merged = pl.concat(out_buf)
        path = f"{out_prefix}worker{worker_id:02d}_part_{part:06d}.parquet"

        merged.write_parquet(path, compression=output_compression)
        print(f"  [worker {worker_id}] wrote {path} ({len(merged):,} rows)", flush=True)

        del merged
        out_buf.clear()
        out_rows = 0
        part += 1

    while True:
        item = file_queue.get()
        if item is None:
            break
        parquet_path = item
        t_file = time.time()
        try:
            t_read = 0.0
            t_prep = 0.0
            t_inf = 0.0
            users_this_file = 0

            t0 = time.time()
            dset = pads.dataset(parquet_path, format="parquet")
            want = [device_id_col] + list(user_cat_cols) + list(user_num_cols) + list(user_multi_cols)
            cols = [c for c in want if c in dset.schema.names]

            seen: set[str] = set()
            pending: pl.DataFrame | None = None

            for batch in _iter_record_batches(dset, cols, infer_stream_batch_rows):
                if max_users_per_file is not None and users_this_file >= int(max_users_per_file):
                    break
                t_read += time.time() - t0
                t0 = time.time()
                bpl = pl.from_arrow(batch)
                bpl = bpl.unique(subset=[device_id_col], keep="first", maintain_order=False)
                bpl = bpl.filter(~pl.col(device_id_col).cast(pl.Utf8).is_in(seen))
                if bpl.is_empty():
                    t0 = time.time()
                    continue
                seen.update(bpl[device_id_col].cast(pl.Utf8).to_list())
                pending = pl.concat([pending, bpl], rechunk=False) if pending is not None else bpl
                del bpl
                t_prep += time.time() - t0

                while pending is not None and len(pending) >= rank_user_batch:
                    if max_users_per_file is not None and users_this_file >= int(max_users_per_file):
                        break
                    t0 = time.time()
                    chunk = _prepare_frame(
                        pending[:rank_user_batch],
                        user_cat_cols,
                        user_num_cols,
                        user_multi_cols,
                    )
                    pending = pending[rank_user_batch:]
                    t_prep += time.time() - t0

                    if max_users_per_file is not None:
                        remaining = int(max_users_per_file) - users_this_file
                        if remaining <= 0:
                            break
                        if len(chunk) > remaining:
                            chunk = chunk[:remaining]

                    t0 = time.time()
                    out_df = _rank_batch_topk(
                        infer_df=chunk,
                        device_id_col=device_id_col,
                        user_cat_cols=user_cat_cols,
                        user_num_cols=user_num_cols,
                        user_multi_cols=user_multi_cols,
                        user_vocabs=user_vocabs,
                        user_multi_vocabs=user_multi_vocabs,
                        multi_max_tokens=multi_max_tokens,
                        expected_vocab_sizes=expected_vocab_sizes,
                        expected_cat_dim=expected_cat_dim,
                        expected_num_dim=expected_num_dim,
                        user_tower_backend=backend,
                        client_emb_t=client_emb_t,
                        client_ids_np=client_ids_np,
                        topk=topk,
                        client_chunk=client_chunk,
                        use_amp=use_amp,
                        amp_dtype_torch=amp_dtype_torch,
                        to_device=to_device,
                        worker_id=worker_id,
                        forward_probe_state=forward_probe_state,
                    )
                    t_inf += time.time() - t0
                    out_buf.append(out_df)
                    out_rows += len(out_df)
                    users_this_file += len(chunk)
                    _flush()
                    del out_df
                    gc.collect()
                t0 = time.time()
                if max_users_per_file is not None and users_this_file >= int(max_users_per_file):
                    break

            if (
                pending is not None
                and len(pending) > 0
                and (max_users_per_file is None or users_this_file < int(max_users_per_file))
            ):
                t0 = time.time()
                chunk = _prepare_frame(
                    pending,
                    user_cat_cols,
                    user_num_cols,
                    user_multi_cols,
                )
                del pending
                t_prep += time.time() - t0

                if max_users_per_file is not None:
                    remaining = int(max_users_per_file) - users_this_file
                    if remaining > 0 and len(chunk) > remaining:
                        chunk = chunk[:remaining]
                    elif remaining <= 0:
                        chunk = chunk[:0]

                t0 = time.time()
                if len(chunk) > 0:
                    out_df = _rank_batch_topk(
                        infer_df=chunk,
                        device_id_col=device_id_col,
                        user_cat_cols=user_cat_cols,
                        user_num_cols=user_num_cols,
                        user_multi_cols=user_multi_cols,
                        user_vocabs=user_vocabs,
                        user_multi_vocabs=user_multi_vocabs,
                        multi_max_tokens=multi_max_tokens,
                        expected_vocab_sizes=expected_vocab_sizes,
                        expected_cat_dim=expected_cat_dim,
                        expected_num_dim=expected_num_dim,
                        user_tower_backend=backend,
                        client_emb_t=client_emb_t,
                        client_ids_np=client_ids_np,
                        topk=topk,
                        client_chunk=client_chunk,
                        use_amp=use_amp,
                        amp_dtype_torch=amp_dtype_torch,
                        to_device=to_device,
                        worker_id=worker_id,
                        forward_probe_state=forward_probe_state,
                    )
                    t_inf += time.time() - t0
                    out_buf.append(out_df)
                    out_rows += len(out_df)
                    users_this_file += len(chunk)
                    _flush()
                    del out_df
                    gc.collect()

            status_queue.put(
                {
                    "worker": worker_id,
                    "file": parquet_path,
                    "users": users_this_file,
                    "read_time": t_read,
                    "preprocess_time": t_prep,
                    "inference_time": t_inf,
                    "total_time": time.time() - t_file,
                    "error": None,
                }
            )
        except Exception as e:
            status_queue.put(
                {
                    "worker": worker_id,
                    "file": parquet_path,
                    "error": repr(e),
                    "traceback": traceback.format_exc(),
                }
            )

    _flush(force=True)

    del client_emb_t, client_ids_np
    if to_device.type == "cuda":
        torch.cuda.empty_cache()
    if user_tower_model is not None:
        user_tower_model.cpu()
        del user_tower_model
    del backend
    gc.collect()
    print(f"  Worker {worker_id} finished. Total parts written: {part}", flush=True)

# ======================================================================================
# Inference orchestrator
# Source: inference/run.py
# ======================================================================================

import multiprocessing
import os
import time
from pathlib import Path



def run_inference_job(cfg: InferJobConfig) -> None:
    """Spawn workers to rank users against precomputed client embeddings (reference flow)."""
    ic = cfg.infer
    runlog = start_run_log(kind="infer", name="two_tower")
    t_start = time.time()
    if ic.debug_cuda:
        os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

    arts = training_artifact_uris(cfg.paths.artifacts_base)
    vocab = load_vocab_artifact_pickle(arts["vocab"])

    user_vocabs_raw = vocab["user_vocabs"]
    user_multi_vocabs_raw = vocab["user_multi_vocabs"]
    user_cat_cols = list(vocab["user_cat_cols"])
    user_num_cols = list(vocab["user_num_cols"])
    user_multi_cols = list(vocab["user_multi_cols"])
    device_id_col = str(vocab["device_id_col"])
    multi_max_tokens = int(vocab["multi_cat_max_tokens"])

    infer_files = list_parquet_inputs(cfg.paths.infer)
    if not infer_files:
        raise FileNotFoundError(f"No inference files under {cfg.paths.infer!r}")
    if ic.max_files is not None:
        infer_files = infer_files[: max(0, int(ic.max_files))]
        if not infer_files:
            raise FileNotFoundError(
                f"infer.max_files={ic.max_files} filtered out all inputs under {cfg.paths.infer!r}"
            )

    out_dir = ic.ranking_output.rstrip("/")
    if not out_dir.startswith("s3://"):
        Path(out_dir).mkdir(parents=True, exist_ok=True)
    out_prefix = out_dir if out_dir.endswith("/") else out_dir + "/"

    num_workers = max(1, int(ic.num_physical_gpus) * max(1, int(ic.workers_per_gpu)))

    print(f"[infer] {num_workers} workers | {len(infer_files)} files | top-{ic.topk_clients}")
    if ic.max_files is not None or ic.max_users_per_file is not None:
        print(f"[infer] limits: max_files={ic.max_files} max_users_per_file={ic.max_users_per_file}")
    print(f"[infer] user tower: {arts['user_tower']}")
    print(f"[infer] client emb: {arts['client_embeddings']}")
    print(f"[infer] output prefix: {out_prefix}")
    runlog.write(
        f"CONFIG infer_path={cfg.paths.infer} artifacts_base={cfg.paths.artifacts_base} "
        f"out={out_prefix} files={len(infer_files)} topk={ic.topk_clients} "
        f"max_files={ic.max_files} max_users_per_file={ic.max_users_per_file}"
    )

    # Jupyter/IPython notebooks + multiprocessing "spawn" can fail because the worker function
    # ends up living in `__main__` (not importable in child processes). On Linux, "fork"
    # avoids pickling the target and is the most compatible for notebooks.
    #
    # For CLI/prod-like execution, we keep the safer default of "spawn".
    start_method = "spawn"
    try:
        in_notebook = bool(getattr(__import__("builtins"), "get_ipython", None))
    except Exception:
        in_notebook = False
    if in_notebook:
        try:
            methods = multiprocessing.get_all_start_methods()
        except Exception:
            methods = []
        if "fork" in methods:
            start_method = "fork"
    ctx = multiprocessing.get_context(start_method)
    file_queue = ctx.Queue()
    status_queue = ctx.Queue()

    for fk in infer_files:
        file_queue.put(fk)
    for _ in range(num_workers):
        file_queue.put(None)

    start_time = time.time()
    processes: list[multiprocessing.Process] = []
    for worker_id in range(num_workers):
        p = ctx.Process(
            target=tt_infer_worker,
            args=(
                worker_id,
                file_queue,
                status_queue,
                arts["user_tower"],
                arts["client_embeddings"],
                out_prefix,
                device_id_col,
                user_cat_cols,
                user_num_cols,
                user_multi_cols,
                user_vocabs_raw,
                user_multi_vocabs_raw,
                multi_max_tokens,
                ic.rank_user_batch,
                ic.topk_clients,
                ic.client_chunk,
                ic.use_amp,
                ic.amp_dtype,
                ic.output_min_rows_per_part,
                ic.output_parquet_compression,
                ic.infer_stream_batch_rows,
                ic.workers_per_gpu,
                ic.max_users_per_file,
                ic.user_tower_backend,
                ic.user_tower_onnx_uri,
                ic.trt_fp16_enable,
                ic.trt_engine_cache_enable,
                ic.trt_engine_cache_path,
            ),
        )
        p.start()
        processes.append(p)

    completed = 0
    total = len(infer_files)
    errors: list[dict] = []
    total_read = total_prep = total_inf = 0.0
    last_hb = time.time()
    hb_every_s = 60.0

    while completed < total:
        try:
            status = status_queue.get(timeout=30)
        except Exception:
            now = time.time()
            if now - last_hb >= hb_every_s:
                runlog.write(f"HEARTBEAT completed_files={completed}/{total} elapsed_min={(now - start_time)/60:.1f}")
                last_hb = now
            continue
        completed += 1
        if status.get("error"):
            errors.append(status)
            print(f"[ERROR] worker={status.get('worker')} file={status.get('file')}")
            print(f"        {status.get('error')}")
            runlog.write(f"FILE_DONE ok=false file={status.get('file')} worker={status.get('worker')} error={status.get('error')}")
        else:
            total_read += float(status.get("read_time", 0))
            total_prep += float(status.get("preprocess_time", 0))
            total_inf += float(status.get("inference_time", 0))
            runlog.write(
                f"FILE_DONE ok=true file={status.get('file')} worker={status.get('worker')} users={status.get('users')} "
                f"t_read={status.get('read_time',0):.2f} t_prep={status.get('preprocess_time',0):.2f} "
                f"t_inf={status.get('inference_time',0):.2f} t_total={status.get('total_time',0):.2f}"
            )
            if completed % 25 == 0 or completed == total:
                elapsed = (time.time() - start_time) / 60
                rate = completed / elapsed if elapsed > 0 else 0
                print(f"Progress: {completed}/{total} | ~{rate:.1f} files/min")
        now = time.time()
        if now - last_hb >= hb_every_s:
            runlog.write(f"HEARTBEAT completed_files={completed}/{total} elapsed_min={(now - start_time)/60:.1f}")
            last_hb = now

    for p in processes:
        p.join()

    elapsed_m = (time.time() - start_time) / 60
    print(
        f"Inference finished in {elapsed_m:.1f} min | errors={len(errors)} | "
        f"cumulative read/prep/inf (s): {total_read:.1f} / {total_prep:.1f} / {total_inf:.1f}"
    )
    if errors:
        runlog.write(f"FINISH ok=false errors={len(errors)} elapsed_s={time.time()-t_start:.1f}")
        raise RuntimeError(f"Inference had {len(errors)} failed file(s); see logs above.")
    runlog.write(f"FINISH ok=true errors=0 elapsed_s={time.time()-t_start:.1f}")


## Run batch inference


In [ ]:
from pathlib import Path

cfg_dict = yaml.safe_load(INFER_CONFIG_YAML)
tmp_cfg = Path('._standalone_infer.yaml').resolve()
tmp_cfg.write_text(yaml.safe_dump(cfg_dict, sort_keys=False), encoding='utf-8')

cfg = load_infer_job_config(tmp_cfg)
run_inference_job(cfg)
